# HW5: Image Captioning

Use this notebook to load your trained checkpoints and explore your model outputs! You can use this notebook to: generated captions for test images, temperature sampling and how it affects output diversity, and attention maps inside your Transformer decoder.

**Make sure you are running this with the `csci1470` conda environment in your kernel!**

In [ ]:
import sys, os, pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

_HOME_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _HOME_DIR not in sys.path:
    sys.path.insert(0, _HOME_DIR)

from assignment import parse_args, load_model

## 1. Configuration

Update the paths below to match where your checkpoints and data are stored.
The defaults match the layout produced by running `assignment.py` with the default `--chkpt_path`.

In [ ]:
DATA_PATH = '../data/data.p'
CHKPT_RNN = '../checkpoints/rnn'
CHKPT_TRA = '../checkpoints/transformer'

print(os.getcwd())

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

with open(DATA_PATH, 'rb') as f:
    data_dict = pickle.load(f)



feat_prep = lambda x: np.repeat(np.array(x).reshape(-1, 2048), 5, axis=0)
img_prep  = lambda x: np.repeat(x / 255.0, 5, axis=0).astype(np.float64)

train_captions  = np.array(data_dict['train_captions'], dtype=np.int64)
test_captions   = np.array(data_dict['test_captions'],  dtype=np.int64)
train_img_feats = feat_prep(data_dict['train_image_features'])
test_img_feats  = feat_prep(data_dict['test_image_features'])
train_images    = img_prep(data_dict['train_images'])
test_images     = img_prep(data_dict['test_images'])
word2idx        = data_dict['word2idx']
idx2word        = data_dict['idx2word']

PAD_ID   = word2idx['<pad>']
START_ID = word2idx['<start>']
END_ID   = word2idx['<end>']
UNK_ID   = word2idx['<unk>']

print(f'Vocab size:            {len(word2idx):,}')
print(f'Train captions shape:  {train_captions.shape}')
print(f'Test  captions shape:  {test_captions.shape}')
print(f'Train img-feat shape:  {train_img_feats.shape}')
print(f'Test  img-feat shape:  {test_img_feats.shape}')
print(f'Stored images:         {test_images.shape[0]} test, {train_images.shape[0]} train (first 100 of each)')

## 2. Dataset Preview

The dataset is **Flickr8k**. We kept a pixel copy of the first 100 images from each split
for visualization. Every image comes with **5 human-written captions**.

In [ ]:
def show_example(images, captions, idx2word, base_idx, title=''):
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.imshow(images[base_idx])
    ax.axis('off')
    ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()
    for j in range(5):
        words = [idx2word[int(i)] for i in captions[base_idx + j]
                 if idx2word[int(i)] not in ('<pad>', '<start>', '<end>')]
        print(f'  [{j+1}] {" ".join(words)}')
    print()

for base in [0, 5, 10]:
    show_example(test_images, test_captions, idx2word, base, title=f'Test image #{base // 5}')

## 3. Loading Your Models

Checkpoints bundle the architecture parameters (`vocab_size`, `hidden_size`, `window_size`),
so the model is rebuilt exactly as it was trained.

If you only trained one model type, that is fine. The notebook will skip visualizations for any model that was not found.

In [ ]:
def try_load(chkpt_dir, model_type, device):
    try:
        args = parse_args(['--type', model_type, '--task', 'test',
                           '--data', DATA_PATH, '--chkpt_path', chkpt_dir])
        model = load_model(args, device)
        model.eval()
        print(f'Loaded {model_type.upper():12s}  '
              f'hidden={model.decoder.hidden_size}  '
              f'vocab={model.decoder.vocab_size}  '
              f'window={model.decoder.window_size}')
        return model
    except FileNotFoundError:
        print(f'No checkpoint found at "{chkpt_dir}/model.pt", skipping {model_type}.')
        return None
    except Exception as e:
        print(f'Error loading {model_type}: {e}')
        return None

rnn_model = try_load(CHKPT_RNN, 'rnn',        DEVICE)
tra_model = try_load(CHKPT_TRA, 'transformer', DEVICE)

In [ ]:
for model, name in [(rnn_model, 'RNN'), (tra_model, 'Transformer')]:
    if model is None:
        continue
    total = sum(p.numel() for p in model.parameters())
    sep = '=' * 55
    print(sep)
    print(f'  {name}: {total:,} parameters')
    print(sep)
    print(model)
    print()

## 4. Generating Captions

We generate captions autoregressively: feed `<start>`, sample the next token from the model's output distribution, append it, repeat until `<end>` or the window limit.

### Temperature

Before sampling, logits are scaled by temperature $T$:

$$p_i = \text{softmax}(\text{logits} \;/\; T)_i$$

- $T \to 0$ (greedy): always picks the highest-probability word, safe but repetitive.
- $T = 1.0$: sample directly from the raw softmax.
- $T > 1.0$: flatter distribution, more surprising, sometimes incoherent.

A good starting point is around `T = 0.1`.

In [ ]:
def gen_caption(model, image_feat, word2idx, temperature=0.1):
    if model is None:
        return '[model not loaded]'

    i2w    = {v: k for k, v in word2idx.items()}
    pad_id = word2idx['<pad>']
    end_id = word2idx['<end>']
    unk_id = word2idx['<unk>']
    window = model.decoder.window_size
    dev    = next(model.parameters()).device

    caption = [word2idx['<start>']]
    img_t   = torch.FloatTensor(image_feat).unsqueeze(0).to(dev)

    model.eval()
    with torch.no_grad():
        while len(caption) < window and caption[-1] != end_id:
            pad_len = window - len(caption)
            cap_in  = caption + [pad_id] * pad_len
            cap_t   = torch.LongTensor(cap_in).unsqueeze(0).to(dev)
            logits  = model(img_t, cap_t)[0, len(caption) - 1]
            probs   = F.softmax(logits / temperature, dim=-1).cpu().numpy()

            token = unk_id
            for _ in range(5):
                token = int(np.random.choice(len(probs), p=probs))
                if token != unk_id:
                    break
            caption.append(token)

    skip = {word2idx['<start>'], end_id, pad_id}
    return ' '.join(i2w[t] for t in caption if t not in skip)

### 4.1 Model vs. Ground Truth

For a set of test images, compare each model's generated caption against the human-written ones.

In [ ]:
N_SHOW = 5
TEMP   = 0.1

for img_idx in range(N_SHOW):
    base      = img_idx * 5
    img_feat  = test_img_feats[base]
    img_pixel = test_images[base]

    fig, ax = plt.subplots(figsize=(4, 3))
    ax.imshow(img_pixel)
    ax.axis('off')
    ax.set_title(f'Test image #{img_idx}', fontsize=10)
    plt.tight_layout(); plt.show()

    print('Ground truth:')
    for j in range(5):
        words = [idx2word[int(i)] for i in test_captions[base + j]
                 if idx2word[int(i)] not in ('<pad>', '<start>', '<end>')]
        print(f'  [{j+1}] {" ".join(words)}')

    if rnn_model:
        print(f'\n  RNN   (T={TEMP}): {gen_caption(rnn_model, img_feat, word2idx, TEMP)}')
    if tra_model:
        print(f'  TRANS (T={TEMP}): {gen_caption(tra_model, img_feat, word2idx, TEMP)}')
    print()

### 4.2 Temperature Exploration

Change `IMG_IDX_TEMP` to pick a different image and observe how temperature shifts the output.

In [ ]:
IMG_IDX_TEMP = 3

base      = IMG_IDX_TEMP * 5
img_feat  = test_img_feats[base]
img_pixel = test_images[base]

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(img_pixel)
ax.axis('off')
ax.set_title(f'Test image #{IMG_IDX_TEMP}', fontsize=10)
plt.tight_layout(); plt.show()

temps = [0.01, 0.1, 0.3, 0.7, 1.5]
print(f'{"Temperature":<12}  {"RNN":<50}  Transformer')
print('-' * 115)
for t in temps:
    rnn_cap = gen_caption(rnn_model, img_feat, word2idx, t) if rnn_model else 'N/A'
    tra_cap = gen_caption(tra_model, img_feat, word2idx, t) if tra_model else 'N/A'
    print(f'  T = {t:<7.2f}  {rnn_cap:<50} | {tra_cap}')

## 5. Transformer Self-Attention

The Transformer decoder uses masked self-attention over the caption: each token can only attend to itself and tokens that came before it (the causal mask enforces this). The result is a $T \times T$ attention matrix where row $i$ shows which earlier tokens token $i$ weighted most heavily when being generated.

Note on cross-attention: because the image is compressed into a single vector, the cross-attention key set has size 1. Softmax over a single value always returns 1.0, so every token attends to the image with equal weight regardless of content. There is nothing to visualize there.

We extract self-attention weights using forward hooks on `AttentionMatrix` modules, so no changes to your model code are needed. This works for both single-head and multi-head implementations.

In [ ]:
def extract_self_attention(model, image_feat_np, caption_ids):
    captured, hooks = {}, []

    for name, module in model.named_modules():
        if module.__class__.__name__ == 'AttentionMatrix':
            def _hook(m, inp, out, n=name):
                captured[n] = out.detach().cpu()
            hooks.append(module.register_forward_hook(_hook))

    if not hooks:
        print('No AttentionMatrix submodules found.')
        print('Attention visualization requires a class named AttentionMatrix.')
        return {}

    model.eval()
    with torch.no_grad():
        dev   = next(model.parameters()).device
        img_t = torch.FloatTensor(image_feat_np).unsqueeze(0).to(dev)
        cap_t = torch.LongTensor(caption_ids).unsqueeze(0).to(dev)
        _     = model(img_t, cap_t)

    for h in hooks:
        h.remove()

    # Keep only self-attention: cross-attention has key-seq length == 1 (single image vector)
    self_attn = {k: v[0].numpy() for k, v in captured.items() if v.shape[2] > 1}
    return self_attn


def get_words(caption_ids, idx2word):
    """Return token strings from a caption, INCLUDING <start>, stopping at <end>/<pad>."""
    words = []
    for i in caption_ids:
        w = idx2word[int(i)]
        if w in ('<end>', '<pad>'):
            break
        words.append(w)
    return words


def plot_self_attention(attn_np, words, title='Self-Attention', skip_start=True):
    """
    Plot a self-attention heatmap.

    skip_start: if True and words[0] == '<start>', omit that row/col so the
                trivial <start>→<start> weight of 1.0 doesn't compress the
                color scale and hide the actual attention patterns.
    """
    offset = 1 if (skip_start and words and words[0] == '<start>') else 0
    display_words = words[offset:]
    T = len(display_words)
    if T == 0:
        return

    mat = attn_np[offset:offset + T, offset:offset + T]
    vmax = float(mat.max()) or 1e-6

    fig, ax = plt.subplots(figsize=(max(5, T * 0.65), max(4, T * 0.6)))
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax.set_xticks(range(T))
    ax.set_xticklabels(display_words, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(T))
    ax.set_yticklabels(display_words, fontsize=9)
    ax.set_xlabel('Attended-to token (key)')
    ax.set_ylabel('Attending token (query)')
    ax.set_title(title, fontsize=11)

    # Annotate cells with numeric values for shorter captions
    if T <= 14:
        threshold = vmax * 0.5
        for i in range(T):
            for j in range(T):
                val = mat[i, j]
                color = 'white' if val > threshold else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=6, color=color)

    plt.tight_layout()
    plt.show()


print('Helpers defined.')

Change `ATTN_INDICES` to pick which test images to visualize.

In [ ]:
ATTN_INDICES = [0, 1, 2, 3, 4]

if tra_model is None:
    print('Transformer not loaded, skipping.')
else:
    for img_idx in ATTN_INDICES:
        base         = img_idx * 5
        attn_feat    = test_img_feats[base]
        attn_caption = test_captions[base][:-1]   # teacher-forcing input (window tokens)
        words        = get_words(attn_caption, idx2word)   # includes <start>

        fig, ax = plt.subplots(figsize=(4, 3))
        ax.imshow(test_images[base])
        ax.axis('off')
        ax.set_title(f'Test image #{img_idx}', fontsize=10)
        plt.tight_layout(); plt.show()

        print('Caption tokens:', ' '.join(words))

        self_attn = extract_self_attention(tra_model, attn_feat, attn_caption)

        if not self_attn:
            continue

        if len(self_attn) > 1:
            avg = np.mean(list(self_attn.values()), axis=0)
            plot_self_attention(avg, words,
                                title=f'Self-Attention for Image #{img_idx}')

        print()

**Reading the plot:** Row $i$ shows which tokens word $i$ attended to when it was generated. Because of the causal mask, the upper triangle should be zero. Look for interesting patterns across different images: does a pronoun attend back to an earlier noun? Do late tokens in a complex scene attend more broadly? Do different images produce noticeably different attention shapes?

## 6. Interactive Head View

An alternative to the static heatmap: `head_view.js` (based on [BertViz](https://github.com/jessevig/bertviz)) renders the same attention weights as an interactive D3 diagram.

- **Hover** over a token on either side to see which tokens it attends to (or receives attention from).
- **Click** a colored square at the top to toggle an attention head on/off.
- **Double-click** a square to isolate that head.

> **Note:** Requires a classic Jupyter Notebook with internet access (loads D3 and jQuery via CDN). It may not render in JupyterLab or VS Code without additional setup.

In [ ]:
import json, uuid
from IPython.display import display, HTML

_HEAD_VIEW_JS = os.path.join(
    os.path.dirname(os.path.abspath('__file__')), '..', 'head_view.js'
)

def show_head_view(attn_weights_dict, words, js_path=_HEAD_VIEW_JS):
    """
    Render self-attention interactively using head_view.js (D3 + jQuery via RequireJS).

    Hover over a token to highlight what it attends to (and what attends to it).
    Click the colored head squares to toggle individual heads.

    Parameters
    ----------
    attn_weights_dict : dict {name: np.ndarray (window, window)}
        One entry per captured AttentionMatrix module; each becomes one head.
    words : list[str]
        Token labels matching the row/col indices (should include '<start>').
    """
    # Skip <start> at position 0 — it attends only to itself and is uninformative
    offset      = 1 if (words and words[0] == '<start>') else 0
    view_words  = words[offset:]
    T           = len(view_words)
    if T == 0:
        print('No content tokens to display.')
        return

    # Build head matrices: list of [T x T] lists, one per captured module
    heads = [mat[offset:offset + T, offset:offset + T].tolist()
             for mat in attn_weights_dict.values()]

    root_id = 'head-view-' + uuid.uuid4().hex[:8]
    params  = {
        'attention': {
            'All Heads': {
                'left_text':  view_words,
                'right_text': view_words,
                'attn': [heads],         # [1 layer][N heads][T][T]
            }
        },
        'default_filter': 'All Heads',
        'root_div_id':    root_id,
        'include_layers': [0],
        'layer':          0,
        'heads':          None,
    }

    try:
        with open(js_path, 'r') as f:
            js_code = f.read().replace('PYTHON_PARAMS', json.dumps(params))
    except FileNotFoundError:
        print(f'head_view.js not found at: {js_path}')
        return

    html = f"""
<div id="{root_id}" style="font-family: Arial, sans-serif; font-size: 14px;">
  <div style="margin-bottom: 6px;">
    Layer: <select id="layer" style="font-size:13px;"></select>
  </div>
  <select id="filter" style="display:none;"></select>
  <div id="vis"></div>
</div>
<script type="text/javascript">
{js_code}
</script>
"""
    display(HTML(html))


print('show_head_view defined.')

In [ ]:
HEAD_VIEW_IDX = 13   # change to any index in ATTN_INDICES

if tra_model is None:
    print('Transformer not loaded, skipping.')
else:
    base         = HEAD_VIEW_IDX * 5
    attn_feat    = test_img_feats[base]
    attn_caption = test_captions[base][:-1]
    words        = get_words(attn_caption, idx2word)

    if HEAD_VIEW_IDX < len(test_images) // 5:
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.imshow(test_images[base])
        ax.axis('off')
        ax.set_title(f'Test image #{HEAD_VIEW_IDX}', fontsize=10)
        plt.tight_layout(); plt.show()

    print('Caption tokens:', ' '.join(words))
    self_attn = extract_self_attention(tra_model, attn_feat, attn_caption)
    if self_attn:
        show_head_view(self_attn, words)

## 7. Try Your Own

Set `IMG_IDX` to any value from 0 to 999 and `TEMP` to your chosen temperature, then run the cell. Images are only stored for indices 0-99, so those will display a picture; higher indices will still generate captions but will not show an image.

In [ ]:
IMG_IDX = 10
TEMP    = 0.1

base     = IMG_IDX * 5
img_feat = test_img_feats[base]

if IMG_IDX < len(test_images) // 5:
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.imshow(test_images[base])
    ax.axis('off')
    ax.set_title(f'Test image #{IMG_IDX}', fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print(f'No stored image for index {IMG_IDX} (only indices 0-99 have images).')

print('Ground truth captions:')
for j in range(5):
    words = [idx2word[int(i)] for i in test_captions[base + j]
             if idx2word[int(i)] not in ('<pad>', '<start>', '<end>')]
    print(f'  [{j+1}] {" ".join(words)}')

print()
if rnn_model:
    print(f'RNN   (T={TEMP}): {gen_caption(rnn_model, img_feat, word2idx, TEMP)}')
if tra_model:
    print(f'TRANS (T={TEMP}): {gen_caption(tra_model, img_feat, word2idx, TEMP)}')